In [ ]:
import os

base = "/workspace/Projects/cultivated-learning"

dirs = [
    f"{base}/core",
    f"{base}/engine",
    f"{base}/data/memory_db",
    f"{base}/data/interaction_log",
    f"{base}/evaluation",
    f"{base}/notebooks",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

for pkg in ["core", "engine", "evaluation"]:
    open(f"{base}/{pkg}/__init__.py", "w").close()

print("✓ Done")

In [ ]:
%%writefile /workspace/Projects/cultivated-learning/engine/inference.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


class InferenceEngine:
    """Wrapper around the base LLM. All model interaction goes through here."""
    
    def __init__(self, model_path, max_context=4096):
        self.model_path = model_path
        self.max_context = max_context
        self.model = None
        self.tokenizer = None
        self.device = None
    
    def load(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path, local_files_only=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            dtype=torch.float16,
            device_map="auto",
            local_files_only=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.device = self.model.device
        
        vram = torch.cuda.memory_allocated(0) / 1e9
        print(f"✓ Loaded {self.model_path}")
        print(f"  VRAM: {vram:.2f} GB")
        print(f"  Max context: {self.max_context} tokens")
        return self
    
    def count_tokens(self, text):
        return len(self.tokenizer.encode(text, add_special_tokens=False))
    
    def generate(self, prompt, max_new_tokens=512, temperature=0.7, top_p=0.9):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_context - max_new_tokens
        ).to(self.device)
        
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        
        prompt_len = inputs["input_ids"].shape[-1]
        new_tokens = output_ids[0][prompt_len:]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        del inputs, output_ids
        torch.cuda.empty_cache()
        
        return response
    
    def generate_structured(self, prompt, max_new_tokens=512, temperature=0.3):
        return self.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.95)
    
    def get_embedding(self, text):
        inputs = self.tokenizer(
            text, return_tensors="pt", truncation=True, max_length=512
        ).to(self.device)A
        
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]
            embedding = last_hidden.mean(dim=1).squeeze()
        
        del inputs, outputs
        torch.cuda.empty_cache()
        
        return embedding.cpu().numpy()

In [ ]:
with open("/workspace/Projects/cultivated-learning/engine/inference.py", "w") as f:
    f.write('''import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


class InferenceEngine:
    """Wrapper around the base LLM. All model interaction goes through here."""
    
    def __init__(self, model_path, max_context=4096):
        self.model_path = model_path
        self.max_context = max_context
        self.model = None
        self.tokenizer = None
        self.device = None
    
    def load(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path, local_files_only=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            dtype=torch.float16,
            device_map="auto",
            local_files_only=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.device = self.model.device
        
        vram = torch.cuda.memory_allocated(0) / 1e9
        print(f"Loaded {self.model_path}")
        print(f"  VRAM: {vram:.2f} GB")
        print(f"  Max context: {self.max_context} tokens")
        return self
    
    def count_tokens(self, text):
        return len(self.tokenizer.encode(text, add_special_tokens=False))
    
    def generate(self, prompt, max_new_tokens=512, temperature=0.7, top_p=0.9):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_context - max_new_tokens
        ).to(self.device)
        
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        
        prompt_len = inputs["input_ids"].shape[-1]
        new_tokens = output_ids[0][prompt_len:]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        del inputs, output_ids
        torch.cuda.empty_cache()
        
        return response
    
    def generate_structured(self, prompt, max_new_tokens=512, temperature=0.3):
        return self.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.95)
    
    def get_embedding(self, text):
        inputs = self.tokenizer(
            text, return_tensors="pt", truncation=True, max_length=512
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]
            embedding = last_hidden.mean(dim=1).squeeze()
        
        del inputs, outputs
        torch.cuda.empty_cache()
        
        return embedding.cpu().numpy()
''')
print("Done")

In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")

from engine.inference import InferenceEngine

engine = InferenceEngine("/workspace/models/results/Mistral-7B-Instruct-v0.3")
engine.load()

response = engine.generate("What is machine learning? Answer in one sentence.")
print(f"Response: {response}")
print(f"Token count: {engine.count_tokens(response)}")

emb = engine.get_embedding("This is a test sentence.")
print(f"Embedding shape: {emb.shape}")

In [ ]:
!pip install chromadb -q

In [ ]:
import chromadb
print(chromadb.__version__)

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/memory_store.py", "w") as f:
    f.write('''import json
import time
import math
import uuid
import chromadb
import numpy as np
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Tuple
from enum import Enum


class MemoryType(Enum):
    EPISODIC = "episodic"
    SEMANTIC = "semantic"
    PROCEDURAL = "procedural"
    REFLECTIVE = "reflective"


@dataclass
class MemoryUnit:
    content: str
    memory_type: MemoryType
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    created_at: float = field(default_factory=time.time)
    last_accessed: float = field(default_factory=time.time)
    access_count: int = 0
    source_interaction_id: str = ""
    salience_score: float = 0.5
    confidence: float = 0.7
    superseded_by: Optional[str] = None
    tags: List[str] = field(default_factory=list)

    def to_metadata(self):
        return {
            "memory_type": self.memory_type.value,
            "created_at": self.created_at,
            "last_accessed": self.last_accessed,
            "access_count": self.access_count,
            "source_interaction_id": self.source_interaction_id,
            "salience_score": self.salience_score,
            "confidence": self.confidence,
            "superseded_by": self.superseded_by or "",
            "tags": json.dumps(self.tags),
        }

    @staticmethod
    def from_chroma(id, document, metadata, embedding=None):
        return MemoryUnit(
            id=id,
            content=document,
            memory_type=MemoryType(metadata["memory_type"]),
            created_at=metadata["created_at"],
            last_accessed=metadata["last_accessed"],
            access_count=metadata["access_count"],
            source_interaction_id=metadata.get("source_interaction_id", ""),
            salience_score=metadata["salience_score"],
            confidence=metadata["confidence"],
            superseded_by=metadata.get("superseded_by", "") or None,
            tags=json.loads(metadata.get("tags", "[]")),
        )


class MemoryStore:
    """Persistent semantic memory with vector search and salience decay."""

    def __init__(self, persist_dir, engine=None, decay_rate=0.01):
        self.engine = engine
        self.decay_rate = decay_rate
        self.client = chromadb.PersistentClient(path=persist_dir)
        self.collection = self.client.get_or_create_collection(
            name="cultivated_memory",
            metadata={"hnsw:space": "cosine"}
        )
        print(f"Memory store initialized: {self.collection.count()} existing memories")

    def store(self, memory: MemoryUnit, embedding=None):
        if embedding is None and self.engine is not None:
            embedding = self.engine.get_embedding(memory.content)

        self.collection.upsert(
            ids=[memory.id],
            documents=[memory.content],
            metadatas=[memory.to_metadata()],
            embeddings=[embedding.tolist()] if embedding is not None else None,
        )
        return memory.id

    def retrieve(self, query_text, top_k=10, min_salience=0.1):
        if self.collection.count() == 0:
            return []

        if self.engine is not None:
            query_embedding = self.engine.get_embedding(query_text)
            results = self.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k * 3, self.collection.count()),
            )
        else:
            results = self.collection.query(
                query_texts=[query_text],
                n_results=min(top_k * 3, self.collection.count()),
            )

        memories = []
        for i in range(len(results["ids"][0])):
            mem = MemoryUnit.from_chroma(
                id=results["ids"][0][i],
                document=results["documents"][0][i],
                metadata=results["metadatas"][0][i],
            )
            if mem.salience_score >= min_salience and mem.superseded_by is None:
                mem.last_accessed = time.time()
                mem.access_count += 1
                self._update_access(mem)
                memories.append(mem)

        memories.sort(key=lambda m: m.salience_score, reverse=True)
        return memories[:top_k]

    def retrieve_by_type(self, memory_type: MemoryType, limit=20):
        results = self.collection.get(
            where={"memory_type": memory_type.value},
            limit=limit,
        )
        memories = []
        for i in range(len(results["ids"])):
            mem = MemoryUnit.from_chroma(
                id=results["ids"][i],
                document=results["documents"][i],
                metadata=results["metadatas"][i],
            )
            memories.append(mem)
        return memories

    def adjust_salience(self, memory_id, delta):
        results = self.collection.get(ids=[memory_id])
        if not results["ids"]:
            return
        metadata = results["metadatas"][0]
        metadata["salience_score"] = max(0.0, min(1.0, metadata["salience_score"] + delta))
        self.collection.update(ids=[memory_id], metadatas=[metadata])

    def decay_pass(self):
        all_memories = self.collection.get()
        updated_ids = []
        updated_metadatas = []
        for i in range(len(all_memories["ids"])):
            metadata = all_memories["metadatas"][i]
            hours_since_access = (time.time() - metadata["last_accessed"]) / 3600
            decay_factor = math.exp(-self.decay_rate * hours_since_access)
            new_salience = metadata["salience_score"] * decay_factor
            if abs(new_salience - metadata["salience_score"]) > 0.001:
                metadata["salience_score"] = new_salience
                updated_ids.append(all_memories["ids"][i])
                updated_metadatas.append(metadata)

        if updated_ids:
            self.collection.update(ids=updated_ids, metadatas=updated_metadatas)
        print(f"Decay pass: updated {len(updated_ids)} of {len(all_memories['ids'])} memories")

    def get_stats(self):
        all_memories = self.collection.get()
        total = len(all_memories["ids"])
        if total == 0:
            return {"total": 0}

        types = {}
        saliences = []
        for m in all_memories["metadatas"]:
            t = m["memory_type"]
            types[t] = types.get(t, 0) + 1
            saliences.append(m["salience_score"])

        return {
            "total": total,
            "by_type": types,
            "avg_salience": sum(saliences) / len(saliences),
            "min_salience": min(saliences),
            "max_salience": max(saliences),
        }

    def _update_access(self, memory: MemoryUnit):
        self.collection.update(
            ids=[memory.id],
            metadatas=[memory.to_metadata()],
        )
''')
print("Done")

In [ ]:
from core.memory_store import MemoryStore, MemoryUnit, MemoryType

memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

# Store some test memories
m1 = MemoryUnit(
    content="User prefers concise, direct answers over lengthy explanations.",
    memory_type=MemoryType.SEMANTIC,
    salience_score=0.9,
    tags=["user_preference", "communication_style"]
)

m2 = MemoryUnit(
    content="User asked about machine learning basics. I gave a detailed overview covering supervised, unsupervised, and reinforcement learning.",
    memory_type=MemoryType.EPISODIC,
    salience_score=0.5,
    tags=["interaction", "machine_learning"]
)

m3 = MemoryUnit(
    content="When discussing technical topics, ground explanations in practical examples rather than abstract theory.",
    memory_type=MemoryType.PROCEDURAL,
    salience_score=0.8,
    tags=["directive", "communication_style"]
)

memory.store(m1)
memory.store(m2)
memory.store(m3)

print(f"\nStats: {memory.get_stats()}")

# Test retrieval
print("\n--- Retrieval test: 'how should I explain things' ---")
results = memory.retrieve("how should I explain things", top_k=3)
for r in results:
    print(f"  [{r.memory_type.value}] (salience: {r.salience_score:.2f}) {r.content[:80]}...")

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/context_assembler.py", "w") as f:
    f.write('''class ContextAssembler:
    """Packs memory, history, directives, and user message into a prompt."""

    def __init__(self, engine, memory_store, max_context=4096, max_response=512):
        self.engine = engine
        self.memory = memory_store
        self.max_context = max_context
        self.max_response = max_response
        self.available_tokens = max_context - max_response

        self.system_prompt = (
            "You are a helpful AI assistant engaged in an ongoing relationship "
            "with your user. You have access to memories from previous interactions. "
            "Use these memories naturally to provide personalized, contextual responses. "
            "Be concise and direct."
        )

    def assemble(self, user_message, conversation_history=None, directives=None):
        sections = []
        token_budget = self.available_tokens

        # 1. System prompt (fixed cost)
        system_section = f"[INST] {self.system_prompt}"
        system_tokens = self.engine.count_tokens(system_section)
        token_budget -= system_tokens

        # 2. User message (fixed cost, reserve first)
        user_section = f"\\nUser: {user_message} [/INST]"
        user_tokens = self.engine.count_tokens(user_section)
        token_budget -= user_tokens

        # 3. Directives (high priority, usually small)
        directive_section = ""
        if directives:
            directive_text = "\\n".join(f"- {d}" for d in directives)
            directive_section = f"\\n\\nActive directives:\\n{directive_text}"
            directive_tokens = self.engine.count_tokens(directive_section)
            if directive_tokens < token_budget * 0.1:
                token_budget -= directive_tokens
            else:
                directive_section = ""

        # 4. Retrieved memories (up to 30% of remaining budget)
        memory_budget = int(token_budget * 0.4)
        memory_section = ""
        if self.memory and self.memory.collection.count() > 0:
            memories = self.memory.retrieve(user_message, top_k=10)
            if memories:
                memory_lines = []
                used_tokens = 0
                for mem in memories:
                    line = f"[{mem.memory_type.value}] {mem.content}"
                    line_tokens = self.engine.count_tokens(line)
                    if used_tokens + line_tokens > memory_budget:
                        break
                    memory_lines.append(line)
                    used_tokens += line_tokens
                if memory_lines:
                    memory_section = "\\n\\nRelevant memories:\\n" + "\\n".join(memory_lines)
                    token_budget -= used_tokens

        # 5. Conversation history (fill remaining space, most recent first)
        history_section = ""
        if conversation_history:
            history_lines = []
            used_tokens = 0
            for turn in reversed(conversation_history):
                line = f"{turn['role'].capitalize()}: {turn['content']}"
                line_tokens = self.engine.count_tokens(line)
                if used_tokens + line_tokens > token_budget:
                    break
                history_lines.insert(0, line)
                used_tokens += line_tokens
            if history_lines:
                history_section = "\\n\\nRecent conversation:\\n" + "\\n".join(history_lines)

        # Assemble final prompt
        prompt = system_section + directive_section + memory_section + history_section + user_section

        return prompt

    def get_token_report(self, prompt):
        total = self.engine.count_tokens(prompt)
        return {
            "prompt_tokens": total,
            "max_context": self.max_context,
            "max_response": self.max_response,
            "remaining_for_response": self.max_context - total,
            "utilization": f"{total / self.available_tokens * 100:.1f}%"
        }
''')
print("Done")

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/interaction_loop.py", "w") as f:
    f.write('''import time
import json
import uuid
from core.memory_store import MemoryUnit, MemoryType


class InteractionLoop:
    """Main loop: assembles context, generates response, stores memories."""

    def __init__(self, engine, memory_store, assembler, log_dir=None):
        self.engine = engine
        self.memory = memory_store
        self.assembler = assembler
        self.log_dir = log_dir
        self.history = []
        self.interaction_count = 0

    def chat(self, user_message):
        self.interaction_count += 1
        interaction_id = str(uuid.uuid4())
        start_time = time.time()

        # Assemble context
        prompt = self.assembler.assemble(
            user_message=user_message,
            conversation_history=self.history,
        )

        # Generate response
        response = self.engine.generate(prompt)

        elapsed = time.time() - start_time

        # Update conversation history
        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": response})

        # Keep history manageable (last 10 turns = 20 messages)
        if len(self.history) > 20:
            self.history = self.history[-20:]

        # Store episodic memory
        episodic = MemoryUnit(
            content=f"User: {user_message}\\nAssistant: {response[:200]}",
            memory_type=MemoryType.EPISODIC,
            source_interaction_id=interaction_id,
            salience_score=0.5,
            tags=["interaction"],
        )
        self.memory.store(episodic)

        # Log interaction
        if self.log_dir:
            self._log(interaction_id, user_message, response, prompt, elapsed)

        return response

    def feedback(self, rating, correction=None):
        """Process explicit feedback on the last interaction."""
        if len(self.history) < 2:
            print("No interaction to rate.")
            return

        last_user = self.history[-2]["content"]
        last_assistant = self.history[-1]["content"]

        # Adjust salience of recent episodic memories
        recent = self.memory.retrieve(last_user, top_k=3)
        delta = (rating - 3) * 0.1  # rating 1-5 maps to -0.2 to +0.2
        for mem in recent:
            self.memory.adjust_salience(mem.id, delta)

        # Store correction as high-salience semantic memory
        if correction:
            correction_mem = MemoryUnit(
                content=f"CORRECTION: {correction}",
                memory_type=MemoryType.SEMANTIC,
                salience_score=0.9,
                confidence=1.0,
                tags=["user_correction", "high_priority"],
            )
            self.memory.store(correction_mem)
            print(f"Stored correction: {correction}")

        print(f"Feedback recorded: rating={rating}, adjusted {len(recent)} memories by {delta:+.2f}")

    def status(self):
        stats = self.memory.get_stats()
        return {
            "interactions": self.interaction_count,
            "history_length": len(self.history),
            "memory": stats,
        }

    def _log(self, interaction_id, user_message, response, prompt, elapsed):
        import os
        os.makedirs(self.log_dir, exist_ok=True)
        log_entry = {
            "id": interaction_id,
            "timestamp": time.time(),
            "user_message": user_message,
            "response": response,
            "prompt_tokens": self.engine.count_tokens(prompt),
            "response_tokens": self.engine.count_tokens(response),
            "elapsed_seconds": round(elapsed, 2),
            "memory_count": self.memory.collection.count(),
        }
        path = os.path.join(self.log_dir, f"{interaction_id}.json")
        with open(path, "w") as f:
            json.dump(log_entry, f, indent=2)
''')
print("Done")

In [ ]:
from core.memory_store import MemoryStore
from core.context_assembler import ContextAssembler
from core.interaction_loop import InteractionLoop

# Initialize the framework
memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

assembler = ContextAssembler(engine=engine, memory_store=memory)

loop = InteractionLoop(
    engine=engine,
    memory_store=memory,
    assembler=assembler,
    log_dir="/workspace/Projects/cultivated-learning/data/interaction_log"
)

print("✓ Cultivated Learning framework initialized")
print(f"  Existing memories: {memory.collection.count()}")
print(f"\n--- First interaction ---\n")

response = loop.chat("What can you tell me about yourself and what you remember?")
print(response)

print(f"\n--- Status ---")
print(loop.status())

In [ ]:
response = loop.chat("I'm working on a project called Contact Front. It's a video game. Remember that.")
print(response)

print(f"\n--- Now asking about it ---\n")

response = loop.chat("What project am I working on?")
print(response)

print(f"\n--- Status ---")
print(loop.status())

In [ ]:
# Give feedback on the last response
loop.feedback(rating=5)

# Store a preference as a high-value memory
loop.feedback(rating=4, correction="When I ask what you remember, just list the facts. Don't elaborate.")

# One more interaction to see if feedback lands
response = loop.chat("What do you remember about me?")
print(response)

print(f"\n--- Final status ---")
print(loop.status())